# Croissance optimale — Cahier de cours

**Goals**
1. Chargez la`OptimalGrowth` package (local `.py` module).
2. Simuler une économie de base.
3. Calculer et *afficher* la fenêtre d'optimisation ** efficace du planificateur**.
4. Fournir **exercices** avec des cellules vides pour les étudiants à compléter.
5. Exécuter **analyses de sensibilité** (paramètres **et** taille de la fenêtre du planificateur).

> Ce cahier est autonome et utilise le fichier local`OptimalGrowth.py` fourni avec le matériel de cours.

## 1) Setup & Imports

In [ ]:
# Run this cell once to check/install the Python packages required for this notebook.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
# If you run this notebook elsewhere, adapt the path so it can find OptimalGrowth.py
import sys, os, importlib, math
from pathlib import Path
import OptimalGrowth  as og


## 2) Paramètres du modèle (`Params`)

In [ ]:
# Inspect default parameters
p = og.Params()
print(p)
print("\nNumber of periods nT =", p.nT, "with step Δ =", p.Delta, "years")
print("Time runs from", p.t0, "to", p.tT)

## 3) Initialiser l'état et déterminer la voie exogène

Avant de résoudre le problème du planificateur, nous devons ** définir l'environnement de l'économie**.
This involves two steps:

1. ** Variables exogènes** — moteurs externes tels que la productivité et la population, qui suivent les chemins que nous spécifions à l'avance.
2. ** Variables d'état initiales** — stocks endogènes comme le capital, qui déterminent d'où vient l'économie.

Nous commençons par les variables exogènes.

### Setting exogenous states

Exogenous state variables are given by $x_t \in \mathbb{R}^{N_x}$: productivity $A_t$, population $L_t$.

They follow an exogenous process:
$$
x_t = g(x_{t-1}), \quad g: \mathbb{R}^{N_x} \to \mathbb{R}^{N_x}.
$$


Dans ce cahier simplifié, nous utilisons deux variables exogènes,$$A_t=L_t=1$$ que nous avons **fixé les deux à 1** (chemins plats) pour isoler la mécanique du noyau.
Nous les stockons dans le$y$ matrix later; other variables are left as `NaN` until solved.

### Setting initial state variables
We also initialize state variables.



In [ ]:
# Initialize the simulation matrix with exogenous drivers and initial states
sim = og.init_states(p)

# Convert to a tidy DataFrame for inspection (optional)
df0 = og.mat_to_df(sim, p)
df0.head()

## 4) Le problème d'optimisation du planificateur

Nous passons maintenant au problème du planificateur central**.
Le planificateur choisit une séquence de contrôles (ici le taux d'épargne*$s_t \in [0,1]$) afin de maximiser l'utilité intertemporelle de la consommation soumise aux lois du mouvement de l'économie.

---

#### Fonction objective

Le planificateur maximise **utilité escomptée**:

$$
\max_{\{s_t\}_{t=0}^{T}} \; \sum_{i=0}^{I^{*}} \frac{1}{(1+\rho)^{i }} \cdot L_{t+i} \cdot U(c_{t+i}),
$$

where:
- $\rho$ est le taux de préférence pour le temps pur,
- $\Delta$ est l'étape temporelle (par exemple en années),
- $I^{*}$ est l'horizon d'optimisation efficace, déterminé par le paramètre de tolérance`toly`,  
- $U(c_t, L_t)$ est la période d'utilité, généralement de **Formulaire CRRA**:
  $$
  U(C_t/L_t) = \frac{(c_t)^{1-\gamma}}{1-\gamma},
  $$
  where $\gamma$ est le coefficient d'aversion relative au risque.

---

#### Constraints

Les choix du planificateur doivent satisfaire à la contrainte **resource** et à la loi **de motion pour le capital**:

- **Resource constraint**:
  $$
  Y_t = C_t + I_t,
  $$
  where $Y_t$ est sortie,$C_t$ consommation, et$I_t$ investment.

- **Capital accumulation**:
  $$
  K_{t+1} = (1 - \delta) K_t + I_t,
  $$
  avec taux d'amortissement$\delta$.

- **Production function**:
  $$
  Y_t = A_t K_t^{\alpha} L_t^{1-\alpha},
  $$
  avec productivité$A_t$, capital share $\alpha$, et le travail$L_t$.

---

#### Effective optimization window

Dans la pratique, l'horizon infini est approché par une fenêtre finie **planner$T_{\text{planner}}$.  
Cette fenêtre est déterminée endogènement par la condition que l'actualisation conduit éventuellement des poids en dessous de la tolérance`toly`:

$$
\text{while } \left(\frac{1}{1+\rho}\right)^{t} > \text{toly}, \quad t \mapsto T_{\text{planner}}.
$$

D'où l'optimisation passe sur un horizon tronqué, assez longtemps pour que les termes futurs au-delà$T_{\text{planner}}$ are negligible.

---

**Interpretation**:  
Le planificateur définit le chemin du taux d'épargne$\{s_t\}$ pour équilibrer la consommation courante* par rapport à l'accumulation future de capital*, en tenant compte de l'actualisation et de l'horizon défini par la fenêtre du planificateur.


In [ ]:
import numpy as np

# Initialize states and simulate with the planner
sim0 = og.init_states(p)
timevec = np.arange(1, p.nT, dtype=int)  # optimization/evolution index

# Bounds and control variable(s): here saving rate s_t in [0,1]
bounds = (0.0, 1.0)
control_id = [p.i_s]

sim_opt = og.run_optimal_policy(sim.copy(), timevec, p, bounds, control_id)

# Define the (effective) planner window size derived from p.rho, p.Delta, and p.toly (module logic).
def planner_window_size(rho, Delta, toly):
    # Reproduce the loop logic in og.run_optimal_policy:
    #   disc starts at 1 and is multiplied by (1/(1+rho))**Delta until it drops below 'toly'.
    if rho <= -1:
        raise ValueError("rho must be > -1")
    disc = 1.0
    Tplanner = 1
    while disc > toly:
        Tplanner += 1
        disc *= (1.0 / (1.0 + rho)) ** Delta
    return Tplanner

T_planner = planner_window_size(p.rho, p.Delta, p.toly)
print(f"Effective planner window (in periods): {T_planner}")
print(f"Time step Δ = {p.Delta} → window ≈ {T_planner * p.Delta} years")

### 3.a) Plots — Optimal Paths

In [ ]:
# Charts rule for this course notebook (matplotlib only; one chart per figure)
import matplotlib.pyplot as plt
import numpy as np

years = sim_opt[:, p.i_time]

# s_t
plt.figure()
plt.plot(years, sim_opt[:, p.i_s], linewidth=2)
plt.title("Optimal saving rate $s_t$")
plt.xlabel("Année"); plt.ylabel("Share"); plt.grid(True)
plt.show()

# K_t
plt.figure()
plt.plot(years, sim_opt[:, p.i_K], linewidth=2)
plt.title("Capital $K_t$")
plt.xlabel("Année"); plt.ylabel("Level"); plt.grid(True)
plt.show()

# Y, C, I (three lines in one figure is acceptable; still one plot figure)
plt.figure()
plt.plot(years, sim_opt[:, p.i_Y], linewidth=2, label="Y")
plt.plot(years, sim_opt[:, p.i_C], linewidth=2, label="C")
plt.plot(years, sim_opt[:, p.i_I], linewidth=2, label="I")
plt.title("Output split: $Y_t = C_t + I_t$")
plt.xlabel("Année"); plt.grid(True); plt.legend()
plt.show()

# c_t per capita
plt.figure()
c = sim_opt[:, p.i_C] / np.maximum(sim_opt[:, p.i_L], 1e-12)
plt.plot(years, c, linewidth=2)
plt.title("Per‑capita consumption $c_t$")
plt.xlabel("Année"); plt.grid(True)
plt.show()

## 4) **Exercises** (remplir les cellules de code)

### Exercice 1 — Reproduire le niveau de référence et vérifier la faisabilité
- Recréez le chemin de base en utilisant vos propres appels de code.
- Verify that resource feasibility holds period by period: \(Y_t = C_t + I_t\).

In [ ]:
# TODO: Your code here.
# Hints:
# p1 = og.Params(); ensure p1.lg is defined; sim1 = og.init_states(p1); ...
pass

### Exercice 2 — Comparez une dépréciation plus élevée`delta`
- Increase `delta` de +5 points de pourcentage (par exemple, de 0,06 à 0,11).
- Resimuler et comparer les chemins horaires de \(s t\), \(K t\) et \(c t\).
- Provide a short economic interpretation.

In [ ]:
# TODO: Your code here.
# Example skeleton:
# p_hi = og.Params(); p_hi.delta = 0.11
# sim_hi = og.run_optimal_policy(og.init_states(p_hi), np.arange(1, p_hi.nT, dtype=int), p_hi, (0,1), [p_hi.i_s])
# Plot comparisons vs. baseline.
pass

### Exercice 3 — Comparer la part de capital plus élevée`alpha`
- Increase `alpha` de 0,30 à 0,45.
- Discutez de la façon dont il modifie l'épargne optimale et l'accumulation de capital.

In [ ]:
# TODO: Your code here.
pass

### Exercice 4 — Ajouter une loi de type DICE sur la croissance démographique

So far, we have treated population as an exogenous flat path ($L_t = 1$).  
Dans les modèles d'évaluation intégrée, la population suit une loi de mouvement** qui reprend l'idée de saturation démographique :

$$
L_{t+1} = L_t^{1-\zeta_L} \cdot L_{\infty}^{\zeta_L} ,
$$

- $L_{\infty}$ est le niveau ** de la population à long terme** (capacité de charge),
- $\zeta_L$ contrôle la vitesse de convergence** vers$L_{\infty}$,  

Questions
- L'étalonnage$\zeta_L$ est actuellement 0, étalonnez-le à 0.02.
- Comparer les résultats avec une population stable.

In [ ]:
# TODO: Your code here.
# You can copy and adapt og.init_states into a new function, then use it in place of the default.
#p.L0   = 1.0           # initial population (scale arbitrary)
#p.Linf = 10500         # Asymptotic population (millions)
#p.lg   = 0             # Population growth rate parameter
pass

#### Interpret your results: does a growing population increase saving? Why? 

> ✍️ You written answer here.

### Exercice 5 — Ajouter une dynamique de type DICE **TFP** avec une croissance en déclin

So far, we kept productivity flat (effectively $A_t \equiv 1$ because `gA = 0`).  
Nous introduisons maintenant une croissance du TFP **variante dans le temps** qui diminue au fil du temps—akin à ce que DICE utilise.

**Loi sur la motion (mise en œuvre`OptimalGrowth.py`):**
$$
A_t \;=\; A_{t-1}\; G_t, 
\qquad\text{avec}\qquad 
G_t \;=\; \frac{1}{\,1 - g_A \exp\!\big(-\delta_A\,\Delta\,(t-1)\big)\,}\,.
$$

De même, le taux de croissance immédiat** est d'environ
$$
g_{A,t} \;\approx\; g_A \exp\!\big(-\delta_A\,\Delta\,(t-1)\big).
$$
Ainsi, ** la croissance initiale** est environ$g_A$, then it **decays at rate** $\delta_A$ toward zero.

**Parameters in `Params`:**
- `A0` — initial TFP level (default $A_0 = 1$; normalization).
- `gA` — **initiale**`0.015*0`, c'est-à-dire **zéro** dans le cahier).
- `deltaA` — ** Taux de diminution** de la croissance du TFP (par an).
- `Delta` — time step (years per period).

---

#### étalonnage suggéré pour cet exercice

Utilisez une croissance initiale positive mais réaliste et une carie lente :

- $A_0 = 1$
- $g_A = 0.015$  Taux de croissance initial du PTF (en %)
- $\delta_A = 0.005$  (la croissance diminue à **0,5 %/an** vers zéro)

**Questions / Tasks**

1. Mettre en œuvre la loi du mouvement ci-dessus (déjà dans le module) en définissant les paramètres en conséquence.
2. Re-simuler le problème du planificateur et **plot$A_t$** et **comparer**:
   - saving rate $s_t$,
   - sortie$Y_t$,
   - consommation par habitant$c_t = C_t/L_t$,
   par rapport au niveau de référence avec TFP constant.


In [ ]:
# 1) Set parameters for TFP growth that declines over time
p_tfp = og.Params()
p_tfp.A0     = 1.0
p_tfp.gA     = 0.015     # ≈ 1.5%/year initial TFP growth
p_tfp.deltaA = 0.005     # growth decay rate per year (toward 0)

#....
pass

3. Discutez: comment la croissance transitoire du TFP modifie-t-elle la trajectoire d'épargne optimale et le bien-être par rapport au TFP plat?

> ✍️ You written answer here.